## ***Innomatics Research Labs : Data Science with Gen AI Internship***

*Below **"Task 2 - Diminos Case Study"** done by Deepak Kaura*

## **Diminos Store - Delivery Time**

### **Problem Statement -**

Kanav has started his own Pizza Store by getting the Franchise from the popular Pizza brand Diminos.

Diminos promises to deliver the pizza order within 31 minutes from the time the order was placed. Otherwise the pizza will be free for the customer.

In order to increase the revenue and profits Kanav is running the store 24 * 7.

Recently Diminos gave a notice to Kanav that they will be measuring their stores' performance by looking at the metric - which is 95th Percentile on Order Delivery time should be less than 31 mins.

Kanav is worried that he might lose the franchise if he is not able to meet the metric and wants your help in order to understand his store's performance so that he can take some actions to prevent his business.  


#### *TASK -*

Assume that you are a freelance data scientist. Help Kanav by analyzing the data and sharing insights to keep his business up and running.

#### ***Dataset Dictionary :-***

| Column               | Meaning                            |
| -------------------- | ---------------------------------- |
| `order_id`           | Unique order identifier            |
| `order_placed_at`    | Timestamp when order was placed    |
| `order_delivered_at` | Timestamp when order was delivered |


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# =====================================================
# 1. LOAD & PREPARE DATA
# =====================================================
df = pd.read_csv("diminos_data.csv")
df["order_placed_at"] = pd.to_datetime(df["order_placed_at"])
df["order_delivered_at"] = pd.to_datetime(df["order_delivered_at"])
df["delivery_time_min"] = (df["order_delivered_at"] - df["order_placed_at"]).dt.total_seconds() / 60
df["order_date"] = df["order_placed_at"].dt.date
df["order_hour"] = df["order_placed_at"].dt.hour
df["day_of_week"] = df["order_placed_at"].dt.day_name()
df["is_late"] = (df["delivery_time_min"] > 31).astype(int)


In [4]:

# =====================================================
# 2. AUTOMATED LOGIC ENGINE (Insights)
# =====================================================
p95_overall = df["delivery_time_min"].quantile(0.95)
late_pct = df["is_late"].mean() * 100
late_by_hour = df[df["is_late"] == 1].groupby("order_hour").size().sort_values(ascending=False)
peak_hours = late_by_hour.head(3).index.tolist()

# Define Store Status
if p95_overall < 28:
    status = "🟢 HEALTHY: Well within SLA."
elif p95_overall <= 31:
    status = "🟡 WARNING: Approaching SLA limit."
else:
    status = "🔴 CRITICAL: SLA Breach detected."


In [11]:

# =====================================================
# 3. INTERACTIVE PLOTLY DASHBOARD
# =====================================================

# =====================================================
# (A) DELIVERY TIME SCORE DISTRIBUTION (REPLACES GAUGE)
# =====================================================

fig_dist = px.histogram(
    df,
    x="delivery_time_min",
    nbins=40,
    title="Delivery Time Distribution (Operational Risk View)",
    labels={"delivery_time_min": "Delivery Time (Minutes)"},
    opacity=0.85
)

# SLA Threshold
fig_dist.add_vline(
    x=31,
    line_dash="dash",
    line_color="red",
    annotation_text="SLA Limit (31 mins)",
    annotation_position="top right"
)

# Warning Threshold
fig_dist.add_vline(
    x=28,
    line_dash="dot",
    line_color="orange",
    annotation_text="Warning Threshold (28 mins)",
    annotation_position="top left"
)

# P95 Marker
fig_dist.add_vline(
    x=p95_overall,
    line_dash="solid",
    line_color="black",
    annotation_text=f"P95 = {p95_overall:.1f} mins",
    annotation_position="bottom right"
)

fig_dist.update_layout(
    width=900,
    height=480,
    bargap=0.1,
    title_x=0.5
)

fig_dist.show()


In [13]:
# (B) DAILY PERFORMANCE TREND
daily_p95 = df.groupby("order_date")["delivery_time_min"].quantile(0.95).reset_index()

fig_trend = px.line(
    daily_p95,
    x="order_date",
    y="delivery_time_min",
    title="Daily Performance Trend (95th Percentile)",
    labels={"delivery_time_min": "Minutes", "order_date": "Date"}
)

fig_trend.add_hline(
    y=31,
    line_dash="dash",
    line_color="red",
    annotation_text="SLA Limit"
)

fig_trend.update_layout(
    width=900,
    height=450,
    title_x=0.5
)

fig_trend.show()


In [14]:
# (C) OPERATIONAL RISK HEATMAP
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

pivot_df = df.groupby(['day_of_week', 'order_hour'])['delivery_time_min'].quantile(0.95).reset_index()
pivot_df['day_of_week'] = pd.Categorical(
    pivot_df['day_of_week'],
    categories=day_order,
    ordered=True
)

pivot_table = pivot_df.pivot(
    index='day_of_week',
    columns='order_hour',
    values='delivery_time_min'
)

fig_heat = px.imshow(
    pivot_table,
    color_continuous_scale='RdYlGn_r',
    title="Operational Risk Matrix (Heatmap)",
    labels=dict(x="Hour of Day", y="Day of Week", color="P95 Mins")
)

fig_heat.update_layout(
    width=900,
    height=500,
    title_x=0.5
)

fig_heat.show()


In [15]:

# =====================================================
# 10. AUTOMATED Q&A ENGINE (DERIVED FROM ANALYTICS)
# =====================================================

sla_limit = 31
warning_limit = 28

# ---- Derived Signals ----
trend_slope = daily_p95["delivery_time_min"].diff().mean()
trend_direction = "INCREASING" if trend_slope > 0 else "STABLE / IMPROVING"

high_risk_zones = (pivot_table > sla_limit).sum().sum()

confidence_score = max(60, int(100 - late_pct))


In [16]:

# =====================================================
# FINAL EXECUTIVE SUMMARY
# =====================================================


print("\n🎯 Final Decision Summary – Prepared by Freelance Data Scientist for Kanav’s DIMINOS Store")

# =====================================================
# Q1. Is the store at risk?
# =====================================================
print("\n❓ Q1. Is the store at risk of SLA violation?")

if p95_overall > sla_limit:
    risk_level = "🔴 CRITICAL"
elif p95_overall >= warning_limit:
    risk_level = "🟡 WARNING"
else:
    risk_level = "🟢 SAFE"

print(f"➡️ Answer:")
print(f"- Risk Level      : {risk_level}")
print(f"- 95th Percentile : {p95_overall:.2f} mins")
print(f"- SLA Limit       : {sla_limit} mins")

# =====================================================
# Q2. What evidence shows delivery risk?
# =====================================================
print("\n❓ Q2. What data indicates delivery risk?")

signals = []

if late_pct > 5:
    signals.append("High percentage of late orders")

if trend_direction == "INCREASING":
    signals.append("Upward trend in daily SLA performance")

if high_risk_zones > 0:
    signals.append("Repeated SLA breaches in day-hour heatmap")

if peak_hours:
    signals.append(f"Late orders concentrated during hours {peak_hours}")

if signals:
    print("➡️ Answer:")
    for i, s in enumerate(signals, 1):
        print(f"{i}. {s}")
else:
    print("➡️ Answer: No strong risk signals detected")

# =====================================================
# Q3. When should Kanav take action?
# =====================================================
print("\n❓ Q3. When should operational action be taken?")

if risk_level != "🟢 SAFE":
    print("➡️ Answer:")
    print(f"- Immediate action during peak hours: {peak_hours}")
    print(f"- Trend status: {trend_direction}")
else:
    print("➡️ Answer:")
    print("- No immediate action required")
    print("- Continue monitoring evening rush")

# =====================================================
# Q4. What actions will reduce delays fastest?
# =====================================================
print("\n❓ Q4. What actions will reduce delivery delays?")

actions = []

if peak_hours:
    actions.append("Add 1 delivery rider during peak hours only")

if late_pct > 3:
    actions.append("Introduce priority dispatch at 20-minute mark")

if high_risk_zones > 0:
    actions.append("Optimize routes for high-risk time windows")

actions.append("Reduce borderline delays (30–35 mins)")

print("➡️ Answer:")
for i, a in enumerate(actions, 1):
    print(f"{i}. {a}")

# =====================================================
# Q5. What is the business impact if ignored?
# =====================================================
print("\n❓ Q5. What happens if delays are not reduced?")

avg_order_value = 250
estimated_daily_loss = int(len(df) * late_pct / 100 * avg_order_value)

print("➡️ Answer:")
print(f"- Late Orders (%)        : {late_pct:.2f}%")
print(f"- Estimated Daily Loss  : ₹{estimated_daily_loss:,}")
print(f"- Franchise Risk Level  : {risk_level}")

# =====================================================
# Q6. How reliable are these insights?
# =====================================================
print("\n❓ Q6. How reliable are these recommendations?")

print("➡️ Answer:")
print(f"- Confidence Score: {confidence_score}%")
print("- Derived from percentile metrics, trend analysis,")
print("  and repeated visual risk patterns")




🎯 Final Decision Summary – Prepared by Freelance Data Scientist for Kanav’s DIMINOS Store

❓ Q1. Is the store at risk of SLA violation?
➡️ Answer:
- Risk Level      : 🟢 SAFE
- 95th Percentile : 27.26 mins
- SLA Limit       : 31 mins

❓ Q2. What data indicates delivery risk?
➡️ Answer:
1. Upward trend in daily SLA performance
2. Repeated SLA breaches in day-hour heatmap
3. Late orders concentrated during hours [1, 11, 16]

❓ Q3. When should operational action be taken?
➡️ Answer:
- No immediate action required
- Continue monitoring evening rush

❓ Q4. What actions will reduce delivery delays?
➡️ Answer:
1. Add 1 delivery rider during peak hours only
2. Introduce priority dispatch at 20-minute mark
3. Optimize routes for high-risk time windows
4. Reduce borderline delays (30–35 mins)

❓ Q5. What happens if delays are not reduced?
➡️ Answer:
- Late Orders (%)        : 3.71%
- Estimated Daily Loss  : ₹139,249
- Franchise Risk Level  : 🟢 SAFE

❓ Q6. How reliable are these recommendations?
